# Qwen2.5-1.5B → counter format → recovery (A100 Colab)

End-to-end witness on a serious donor: warm-start the 1.5B into 6-bit counter synapses
(`counter_packed`, fused Triton update WITH the row clip), then recover quality by
KD-distillation on a pretraining-like EN/RU/code/math mix.

**Setup (one time):**
1. Runtime → Change runtime type → **A100 GPU**.
2. Add Colab secrets (key icon): `KAGGLE_USERNAME`, `KAGGLE_KEY` (from kaggle.json).
3. Run all. Checkpoints land on Drive → a dropped session resumes from the last one.

Budget guide: `MAX_HOURS=0.8` for the pilot (~9 compute units), `10.5` for the main run.

In [ ]:
# --- data + code from the private Kaggle dataset -----------------------------
import os, subprocess
from google.colab import userdata, drive

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
DATASET = "lirovkharki/mn-recovery-15b-corpus"

if not os.path.exists("/content/data/manifest.json"):
    subprocess.run(["kaggle", "datasets", "download", DATASET, "-p", "/content/data", "--unzip"], check=True)
import sys
sys.path.insert(0, "/content/data/mnsrc")   # memory_native/ (Kaggle auto-extracts the tarball)
sys.path.insert(0, "/content/data")         # recovery_session.py rides in the dataset
print(os.listdir("/content/data"))

In [ ]:
# --- Drive for checkpoints ----------------------------------------------------
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/mn_recovery_15b"
os.makedirs(WORK, exist_ok=True)
CKPT, LOG = f"{WORK}/ckpt.pt", f"{WORK}/log.jsonl"
print(WORK)

In [ ]:
# --- config --------------------------------------------------------------------
MODEL = "Qwen/Qwen2.5-1.5B"
SEQ, BATCH = 1024, 8            # A100-40GB; drop to (1024, 4) on OOM
MAX_HOURS = 0.8                 # PILOT. Main run: 10.5
KD_ALPHA, CE_ALPHA, TEMPERATURE = 1.0, 0.3, 2.0
LR, GRAD_CLIP = 3e-4, 1.0       # AdamW on the fp slice (embeddings/norms/biases)
COUNTER_LR, LOCAL_GRAD_CLIP, THRESHOLD = 0.008, 1.0, 0.5   # the stable recipe
EVAL_EVERY = CKPT_EVERY = 400
SEED = 0

In [ ]:
# --- build: teacher + counter student -------------------------------------------
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from memory_native.donor.qwen import qwen_to_counter
from recovery_session import DomainMix, run_session, eval_all

torch.manual_seed(SEED)
DEV = "cuda"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
teacher = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.bfloat16, attn_implementation="sdpa").to(DEV).eval()
student = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.bfloat16, attn_implementation="sdpa").to(DEV)
report = qwen_to_counter(student, kind="counter_packed", threshold_ratio=THRESHOLD,
                         lr=COUNTER_LR, local_grad_clip=LOCAL_GRAD_CLIP,
                         cache_mode="int8")   # int8 T-cache: half the fp16 cache memory
print(report)
mix = DomainMix("/content/data", seq=SEQ, batch=BATCH, seed=SEED)
print("epoch steps:", mix.epoch_steps())
print(f"GPU mem after build: {torch.cuda.memory_allocated()/2**30:.1f} GiB")

In [ ]:
# --- teacher baseline (once) + train with resume --------------------------------
val = mix.val_batches(DEV)
if not os.path.exists(f"{WORK}/teacher_baseline.json"):
    import json as _json
    from memory_native.eval import perplexity
    base = {f"ppl_{n}": perplexity(teacher, b) for n, b in val.items()}
    _json.dump(base, open(f"{WORK}/teacher_baseline.json", "w"))
    print("teacher baseline:", base)

final = run_session(
    student=student, teacher=teacher, tokenizer=tokenizer, mix=mix, device=DEV,
    steps=mix.epoch_steps(), ckpt_path=CKPT, log_path=LOG,
    kd_alpha=KD_ALPHA, ce_alpha=CE_ALPHA, temperature=TEMPERATURE,
    lr=LR, grad_clip=GRAD_CLIP, eval_every=EVAL_EVERY, ckpt_every=CKPT_EVERY,
    max_hours=MAX_HOURS)
final

In [ ]:
# --- report ----------------------------------------------------------------------
import json as _json
base = _json.load(open(f"{WORK}/teacher_baseline.json"))
warm = next(_json.loads(l) for l in open(LOG, encoding="utf-8") if _json.loads(l).get("phase") == "warm")
print(f"{'domain':8s} {'fp teacher':>12s} {'warm-start':>12s} {'recovered':>12s}")
for d in ("en", "ru", "code", "math"):
    print(f"{d:8s} {base['ppl_'+d]:12.1f} {warm['ppl_'+d]:12.1f} {final['ppl_'+d]:12.1f}")
print()
for k, v in final["gen"].items():
    print(f"--- {k}
{v}
")